# Capstone — FlyRank Content Refresh Prioritization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This notebook brings the FlyRank lane together: the content-refresh question, time-aware feature design, baseline/model comparison, limitations, and the W07 action playbook. The goal is decision support for a human content reviewer, not automatic content changes.

## 1. Question

### FlyRank case-study question

When a content team has too many pages to review manually, can recent search-performance signals help identify which pages deserve attention first?

The decision this supports is **review prioritization**: create a focused queue using observable search telemetry, while keeping the final content decision with a human reviewer.

In [ ]:
# Research framing
QUESTION = "Can recent search-performance signals support content-review prioritization?"
DECISION = "Which pages should a human reviewer investigate first?"
print(QUESTION)
print(DECISION)

## 2. Data

The modeling panel is the processed March 2026 `refresh_feature_vector.csv` generated during the internship workflow. It contains 3,434,323 observations. The broader release is much larger, but this notebook reports results from the processed modeling panel rather than claiming to train directly on the full release.

The public-safe feature set contains seven-day historical clicks, impressions, average position, CTR, and a weekend indicator. The target is next-day clicks. Client names, URLs, page titles, keywords, and private queries are excluded from the public artifact.

In [ ]:
import pandas as pd

FEATURES = [
    'clicks_7d_avg',
    'impressions_7d_avg',
    'position_7d_avg',
    'ctr_7d',
    'is_weekend'
]
TARGET = 'target_next_day_clicks'
print('Features:', FEATURES)
print('Target:', TARGET)

## 3. Methodology

The target is numeric, so this is a regression problem. I compare a transparent seven-day-average baseline with Decision Tree, Random Forest, and Gradient Boosting regressors.

Validation is chronological: March 1–24, 2026 for training and March 25–30, 2026 for validation. This avoids randomly mixing future observations into training.

Leakage control is part of the method: only information available by the prediction date belongs in the feature vector. Future target information and intentionally constructed future features are excluded.

In [ ]:
# Reproducible split design
TRAIN_END = '2026-03-24'
VALIDATION_START = '2026-03-25'
VALIDATION_END = '2026-03-30'
BASELINE = 'clicks_7d_avg'
MODELS = ['Decision Tree', 'Random Forest', 'Gradient Boosting']
print(f'Train through {TRAIN_END}')
print(f'Validate {VALIDATION_START} through {VALIDATION_END}')
print('Baseline:', BASELINE)
print('Models:', ', '.join(MODELS))

## 4. Results vs baseline

The held-out validation results are:

| Method | MAE ↓ | RMSE ↓ | R² ↑ |
|---|---:|---:|---:|
| **7-day average baseline** | **0.2094** | **0.7904** | **0.6852** |
| Decision Tree | 0.2266 | 0.8762 | 0.6132 |
| Random Forest | 0.2248 | 0.8665 | 0.6217 |
| Gradient Boosting | 0.2251 | 0.8217 | 0.6597 |

The honest result is that none of the tested ML models beat the simple historical baseline on this validation window.

In [ ]:
results = pd.DataFrame({
    'method': ['7-day average baseline', 'Decision Tree', 'Random Forest', 'Gradient Boosting'],
    'MAE': [0.2094, 0.2266, 0.2248, 0.2251],
    'RMSE': [0.7904, 0.8762, 0.8665, 0.8217],
    'R2': [0.6852, 0.6132, 0.6217, 0.6597]
})
results

### Validation chart

![Validation RMSE](../../paper/assets/rmse.png)

Lower RMSE is better. The baseline remains the strongest result.

## 5. Limitations

- This is observational search-performance data, not an intervention experiment.
- Prediction does not establish that refreshing a page causes more clicks.
- The system does not model or reverse-engineer a search ranking algorithm.
- The reported holdout is only March 25–30, 2026.
- The current feature set is intentionally small and public-safe.
- The baseline currently beats the tested ML models, so model complexity is not justified yet.

In [ ]:
LIMITATIONS_CHECK = {
    'causal_claim': False,
    'automatic_content_changes': False,
    'future_information_used': False,
    'validation_is_chronological': True
}
LIMITATIONS_CHECK

## 6. Ranked recommendations

The W07 action playbook turns the modeling lane into a human-review queue.

1. **HIGH_IMPRESSIONS** — investigate pages receiving substantial visibility but not necessarily converting that visibility into clicks.
2. **LOW_CTR** — review title/snippet alignment and whether the result matches search intent.
3. **WEAKER_POSITION** — investigate relevance, content quality, and on-page signals.
4. **LOW_RECENT_CLICKS** — monitor the recent trend and investigate possible decline.
5. **MODEL_PRIORITY** — use predicted demand as a supporting signal, not as an automatic decision.

In [ ]:
reason_to_action = {
    'HIGH_IMPRESSIONS': 'Review whether visibility is converting into clicks.',
    'LOW_CTR': 'Review title/snippet alignment and search intent.',
    'WEAKER_POSITION': 'Review relevance, content quality and on-page signals.',
    'LOW_RECENT_CLICKS': 'Monitor recent trend and investigate decline.',
    'MODEL_PRIORITY': 'Send for general human review.'
}
for reason, action in reason_to_action.items():
    print(f'{reason}: {action}')

## 7. Artifacts the paper embeds

The deployed paper contains the validation RMSE and R² charts and the case-study narrative. The public repository keeps the workflow and public-safe artifacts rather than private client-level content.

In [ ]:
from pathlib import Path

for artifact in ['paper/assets/rmse.png', 'paper/assets/r2.png']:
    print(artifact, 'exists:', Path(artifact).exists())

## 8. 5-Minute Week-8 Demo Outline

**Question (0:00–0:45)**

Can recent search-performance signals help a content team decide which pages deserve review first? The practical FlyRank problem is prioritization: when a large content portfolio cannot be reviewed page-by-page, use observable search telemetry to create a focused, human-reviewed queue.

**Method (0:45–2:00)**

Use the processed March 2026 `refresh_feature_vector.csv` panel. Build seven-day historical features for clicks, impressions, average position and CTR, plus a weekend indicator; predict next-day clicks; compare a simple seven-day-average baseline with Decision Tree, Random Forest and Gradient Boosting regressors using a chronological March 1–24 training window and March 25–30 validation window. Explicitly exclude future information and check for leakage before interpreting model performance.

**One chart (2:00–3:00)**

Show the validation RMSE comparison from the paper (`paper/assets/rmse.png`). Lower is better, and the seven-day-average baseline is the strongest result.

**One honest result (3:00–4:00)**

The baseline achieved RMSE ≈ 0.7904, while Decision Tree, Random Forest and Gradient Boosting were higher at ≈ 0.8762, 0.8665 and 0.8217. The tested ML models therefore did **not** improve on the simple historical baseline on this validation window.

**One recommendation (4:00–5:00)**

Keep the simple baseline as the current prediction layer and use the W07 action playbook as decision support: prioritize pages with high impressions, low CTR, weaker average position or declining recent clicks, then let a human decide what to investigate. Test additional time windows and richer features before replacing the baseline.

**Closing line:** The outcome is not an automatic content optimizer; it is a reproducible, public-safe workflow that turns search telemetry into a ranked review queue while keeping prediction, recommendation, and causality distinct.

## 9. Shareable Cuts

### Short social post

I built a time-aware ML workflow for a real content-refresh prioritization problem: how can search-performance signals help a team decide which pages to review first? I used seven-day historical clicks, impressions, position and CTR features, compared tree-based regressors against a simple historical baseline, and used a chronological validation split with leakage checks. The most useful finding was also the most honest one: the simple baseline beat the tested ML models on the March 25–30 validation window, so the current system is better framed as decision support than as an automatic optimizer.

### Employer-facing summary

I built a time-aware machine-learning pipeline and decision-support workflow for FlyRank's content-refresh prioritization problem, using a processed March 2026 panel of 3.43M observations derived from anonymized search-performance data. I engineered seven-day click, impression, position and CTR features, tested Decision Tree, Random Forest and Gradient Boosting regression models, and evaluated them with a chronological holdout and leakage checks. The simple seven-day-average baseline achieved the best validation RMSE (≈0.7904), showing that the current feature set did not yet justify replacing a strong baseline and motivating a human-reviewed prioritization workflow rather than an automatic content-change system.

## Self-check

- [x] Research question tied to the FlyRank content problem
- [x] Data and feature framing documented
- [x] Time-aware validation and leakage framing documented
- [x] Baseline and model results recorded honestly
- [x] Limitations and recommendations included
- [x] Week-8 demo outline included
- [x] Social post and employer-facing summary included
- [ ] Run all cells in Colab with the local processed dataset before presenting